# Lezione 30 — L'attenzione come recupero morbido

> **Modulo:** Transformer e modello open · **Tempo stimato:** 25 minuti
> **Prerequisiti:** Lezione 18 (similarita' coseno), Lezione 28 (retrieval ibrido).
> Apre la Fase 5: dai meccanismi di memoria ai Transformer.
>
> Obiettivo pratico unico: implementare da zero, in NumPy, la
> **scaled dot-product attention** e usarla come forma *morbida* e
> differenziabile di recupero sulle memorie del progetto.

## Parte 1 — Teoria: query, chiavi, valori

Nella Lezione 28 il retrieval era una scelta **netta**: date le memorie,
ne selezioni le prime *k* per punteggio. L'attenzione fa la stessa cosa in
modo **morbido**: invece di scegliere un sottoinsieme, assegna a *ogni*
memoria un peso tra 0 e 1 (i pesi sommano a 1) e restituisce una **media
pesata** dei loro contenuti.

Tre ruoli, presi in prestito dal linguaggio dell'information retrieval:

- **Query (Q)**: cosa sto cercando (la memoria/domanda corrente).
- **Chiavi (K)**: l'indice su cui confronto la query (una per memoria).
- **Valori (V)**: il contenuto che voglio recuperare (una per memoria).

La formula della *scaled dot-product attention* (Vaswani et al., 2017,
Sez. 3.2.1) e':

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Il prodotto $QK^\top$ misura quanto ogni chiave somiglia alla query (come
il coseno della Lezione 18, ma senza normalizzare). La divisione per
$\sqrt{d_k}$ evita che, con dimensioni grandi, i punteggi diventino enormi
e spingano la softmax in una zona a gradiente quasi nullo. La softmax
trasforma i punteggi in pesi che sommano a 1.

In [1]:
import numpy as np

rng = np.random.default_rng(30)


def softmax(scores, axis=-1):
    # sottraggo il massimo per stabilita' numerica: non cambia il risultato
    scores = scores - np.max(scores, axis=axis, keepdims=True)
    exp = np.exp(scores)
    return exp / np.sum(exp, axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V):
    d_k = K.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)      # (n_query, n_key)
    weights = softmax(scores, axis=-1)   # ogni riga somma a 1
    output = weights @ V                 # (n_query, d_v)
    return output, weights


# verifica minima: i pesi sommano a 1
Q = rng.normal(size=(2, 4))
K = rng.normal(size=(5, 4))
V = rng.normal(size=(5, 3))
out, w = scaled_dot_product_attention(Q, K, V)
print("shape output:", out.shape, "| shape pesi:", w.shape)
print("somma dei pesi per query:", np.round(w.sum(axis=1), 6))

shape output: (2, 3) | shape pesi: (2, 5)
somma dei pesi per query: [1. 1.]


Leggi l'output: i pesi hanno forma `(n_query, n_key)` e **ogni riga somma
a 1**. Questo e' cio' che distingue l'attenzione dal retrieval netto: nessuna
memoria viene scartata, ma alcune pesano molto piu' di altre.

## Parte 2 — Attenzione sulle memorie reali

Non abbiamo un modello di embedding addestrato in questo ambiente, quindi
costruiamo un **embedding deterministico** semplice (hashing delle parole in
un vettore a dimensione fissa). Non e' semanticamente ricco come quelli
delle Lezioni 16–20, ma e' riproducibile e serve a mostrare il meccanismo:
memorie che condividono parole avranno chiavi piu' vicine.

In [2]:
import pandas as pd
import re
from pathlib import Path

train = pd.read_csv(Path('..') / 'datasets' / 'processed' / 'memory_train.csv')

DIM = 32


def embed(testo, dim=DIM):
    vec = np.zeros(dim)
    for parola in re.findall(r"[a-zA-Z]+", str(testo).lower()):
        # hash deterministico della parola -> un indice fisso del vettore
        # (non uso hash() perche' e' randomizzato tra processi)
        h = int.from_bytes(parola.encode(), 'little') % dim
        vec[h] += 1.0
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec


# prendo un piccolo campione di memorie come "banca" da cui recuperare
banca = train.head(6).reset_index(drop=True)
K = np.vstack([embed(t) for t in banca['text']])   # chiavi
V = K.copy()                                        # i valori qui coincidono con le chiavi
for i, t in enumerate(banca['text']):
    print(f"[{i}] {t}")

[0] The user likes walking meetings.
[1] The user prefers morning sessions for il progetto TensorFlow.
[2] Lucia visited Bologna for the weekend.
[3] Marco met a friend in Milano.
[4] Elena visited Glasgow for the weekend.
[5] The user likes walking meetings.


Ora formulo una **query**: una nuova memoria in arrivo. L'attenzione mi
dira' su quali memorie della banca il modello si concentrerebbe.

In [3]:
query_text = "The user prefers morning sessions for the project."
Q = embed(query_text).reshape(1, -1)

out, w = scaled_dot_product_attention(Q, K, V)
pesi = w[0]

ordine = np.argsort(pesi)[::-1]
print("Query:", query_text)
print()
print("Peso di attenzione per memoria (ordinato):")
for i in ordine:
    print(f"  peso={pesi[i]:.3f}  [{i}] {banca['text'].iloc[i]}")

Query: The user prefers morning sessions for the project.

Peso di attenzione per memoria (ordinato):
  peso=0.180  [1] The user prefers morning sessions for il progetto TensorFlow.
  peso=0.167  [5] The user likes walking meetings.
  peso=0.167  [0] The user likes walking meetings.
  peso=0.162  [4] Elena visited Glasgow for the weekend.
  peso=0.162  [2] Lucia visited Bologna for the weekend.
  peso=0.162  [3] Marco met a friend in Milano.


Leggi i pesi: le memorie che condividono parole con la query (`user`,
`prefers`, `morning`, `sessions`, `project`) ricevono il peso maggiore. Il
vettore `out` e' la loro **media pesata** — un unico vettore di contesto che
riassume "cio' che la banca sa" di rilevante per la query.

### Esercizio

Scrivi una query diversa — una memoria *episodica* su un viaggio (es. una
frase che contiene `Glasgow`) — calcola i pesi di attenzione sulla stessa
banca e osserva se l'attenzione si sposta. Prova prima da solo, poi
confronta con la soluzione.

In [4]:
# Scrivi qui la tua query e calcola i pesi di attenzione.
# query_mia = "..."
# Q_mia = embed(query_mia).reshape(1, -1)
# _, w_mia = scaled_dot_product_attention(Q_mia, K, V)
# print(w_mia[0])

### Soluzione spiegata

In [5]:
query_mia = "Marco visited Glasgow with his son in July."
Q_mia = embed(query_mia).reshape(1, -1)
_, w_mia = scaled_dot_product_attention(Q_mia, K, V)

pesi_mia = w_mia[0]
i_top = int(np.argmax(pesi_mia))
print("Query episodica:", query_mia)
print(f"Memoria piu' attesa: peso={pesi_mia[i_top]:.3f} -> [{i_top}] {banca['text'].iloc[i_top]}")
print("Distribuzione pesi:", np.round(pesi_mia, 3))
# La distribuzione e' piu' piatta: nessuna memoria della banca parla di viaggi,
# quindi l'attenzione non trova un forte aggancio e si distribuisce.

Query episodica: Marco visited Glasgow with his son in July.
Memoria piu' attesa: peso=0.169 -> [4] Elena visited Glasgow for the weekend.
Distribuzione pesi: [0.166 0.165 0.165 0.169 0.169 0.166]


## Parte 3 — Il progetto: Memory AI Lab, passo 30

Aggiungo al progetto una funzione di **recupero per attenzione**: data una
query e una banca di memorie, restituisce (a) il vettore di contesto e (b) le
memorie ordinate per peso. E' il ponte concettuale verso i Transformer: la
stessa operazione, ripetuta e addestrata, diventa il cuore del modello.

In [6]:
def recupero_per_attenzione(query_text, banca_df, top_n=3):
    K = np.vstack([embed(t) for t in banca_df['text']])
    V = K.copy()
    Q = embed(query_text).reshape(1, -1)
    contesto, w = scaled_dot_product_attention(Q, K, V)
    pesi = w[0]
    ordine = np.argsort(pesi)[::-1][:top_n]
    ranked = [(float(pesi[i]), banca_df['text'].iloc[i]) for i in ordine]
    return contesto[0], ranked


contesto, ranked = recupero_per_attenzione(query_text, banca, top_n=3)

# controllo che non regredisca: il contesto e' un vettore della dimensione attesa
# e i pesi restituiti sono ordinati in modo decrescente.
assert contesto.shape == (DIM,)
assert all(ranked[i][0] >= ranked[i + 1][0] for i in range(len(ranked) - 1))

print("Top memorie per attenzione:")
for peso, testo in ranked:
    print(f"  {peso:.3f}  {testo}")
print("\nnorma del vettore di contesto:", round(float(np.linalg.norm(contesto)), 4))

Top memorie per attenzione:
  0.180  The user prefers morning sessions for il progetto TensorFlow.
  0.167  The user likes walking meetings.
  0.167  The user likes walking meetings.

norma del vettore di contesto: 0.7303


## Riepilogo (max 8 punti)

1. L'attenzione e' un **recupero morbido**: pesi tra 0 e 1 che sommano a 1,
   non una selezione netta.
2. Tre ruoli: **query** (cosa cerco), **chiavi** (indice), **valori**
   (contenuto).
3. Formula: $\text{softmax}(QK^\top/\sqrt{d_k})\,V$.
4. Il prodotto $QK^\top$ e' una similarita' non normalizzata (parente del
   coseno della Lezione 18).
5. La divisione per $\sqrt{d_k}$ tiene la softmax fuori dalla zona a
   gradiente quasi nullo.
6. La softmax rende i punteggi una distribuzione di probabilita'.
7. L'output e' la **media pesata** dei valori: un vettore di contesto.
8. E' l'operazione elementare da cui, ripetuta e addestrata, nasce il
   Transformer (Lezione 32).

## Quiz

1. Perche' si divide per $\sqrt{d_k}$ e non per $d_k$?
2. Cosa cambia tra il retrieval della Lezione 28 e l'attenzione di oggi?
3. Se tutti i punteggi $QK^\top$ fossero uguali, come sarebbero i pesi?

*(Risposte: 1. per compensare la crescita dei prodotti scalari ~ $\sqrt{d_k}$
e mantenere gradienti utili; 2. netto/top-k vs. media pesata differenziabile
su tutte le memorie; 3. uniformi, cioe' 1/n per ciascuna memoria.)*